# Regime Engine v2 — Calibration Q5

**Round 5** resets the calibration baseline by fixing a workflow bug: `run_regime_v2_calibration()` was not loading or passing the macro-method config from [`configs/regime_detection/fred_series.yml`](../../configs/regime_detection/fred_series.yml).

That meant calibration runs used `MacroRegimeConfig()` defaults instead of the Q2/Q3 macro engine settings, especially the intended `compression: tanh` path. The result was a misleading calibration baseline where macro concept scores could sit at the clip cap and dominate anchor-period interpretation.

**Files changed in this round**:

1. [`market_helper/workflows/regime_calibration.py`](../../market_helper/workflows/regime_calibration.py) now calls `load_macro_regime_config(specs_path)` and passes `macro_method_config` into `run_regime_engine_v2()`.
2. [`tests/unit/workflows/test_regime_calibration.py`](../../tests/unit/workflows/test_regime_calibration.py) adds a regression test that captures the workflow call and asserts the loaded macro config is forwarded.

**Evidence inputs**:

* Generated summary: `data/artifacts/regime_detection/calibration/regime_v2_calibration_summary.json`
* Generated questions notebook: `notebooks/regime_detection/regime_v2_calibration_questions.ipynb`
* Local command: `conda run -n py313 python -m market_helper.cli.main regime-calibrate`


## Root Cause

| item | Intended calibration behavior | Pre-Q5 behavior | Q5 behavior |
|---|---|---|---|
| Macro method config source | Load `engine:` block from `fred_series.yml` | Omitted; workflow fell back to `MacroRegimeConfig()` defaults | Explicitly loaded via `load_macro_regime_config(specs_path)` |
| Macro compression | Reuse Q2/Q3 `compression: tanh`, `compression_k: 2.0` | `compression = none`, so threshold-normalized concepts could sit at raw clip values such as `3.0` | Compression path is active during calibration, matching the intended engine contract |
| What this invalidated | Threshold/weight review should be based on the same macro scale as production v2 | Calibration overstated macro amplitude and disagreement | Baseline now matches the calibrated macro latent space |

**Interpretation**: the immediate problem was not necessarily `regime_engine.yml` thresholds. The calibration baseline itself was wrong, so any config tuning on top of it would have been difficult to trust.


## Anchor Comparison — Before vs After The Fix

| anchor | pre-Q5 majority | post-Q5 majority | pre macro G/I | post macro G/I |
|---|---|---|---|---|
| 2008-09 GFC | Slowdown / Deflationary Slowdown | Slowdown / Deflationary Slowdown + Stress Overlay | `-2.77 / -0.56` | `-0.87 / -0.19` |
| 2009-10 Recovery | Goldilocks / Expansion | Neutral/Mixed Growth / Neutral/Mixed Inflation | `-0.51 / -1.10` | `-0.15 / -0.35` |
| 2020 COVID Shock | Slowdown / Deflationary Slowdown | Slowdown / Deflationary Slowdown + Stress Overlay | `-1.14 / -1.10` | `-0.36 / -0.36` |
| 2020 H2-2021 Reopening | Reflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.22 / -0.29` | `0.11 / -0.14` |
| 2022 Inflation Shock / Tightening | Reflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.67 / 1.88` | `0.31 / 0.57` |
| 2023-24 Disinflation / Soft Landing | Neutral/Mixed Growth / Up Inflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `-0.01 / 1.64` | `-0.01 / 0.53` |
| 2025 April Liberation Day Tariff Shock | Neutral/Mixed Growth / Neutral/Mixed Inflation | Neutral/Mixed Growth / Neutral/Mixed Inflation | `0.11 / 0.46` | `0.05 / 0.18` |

The most important shift is not the labels themselves; it is the **macro amplitude reset**. Once macro compression is applied during calibration, the engine no longer looks permanently inflation-hot in 2022-24 by construction.


## What The Fixed Baseline Says

1. **Crisis windows still hold up**. GFC and COVID remain slowdown/stress regimes even after macro magnitudes are compressed. That means the engine's core crisis recognition is not dependent on the buggy high-magnitude macro baseline.

2. **The earlier 2022 / 2023-24 inflation readings were overstated in calibration**. Under the fixed baseline, 2022 and 2023-24 stop defaulting to `Reflation` or `Up Inflation` simply because macro inflation concepts were sitting near the raw clip cap.

3. **The real next problem is now visible**: recovery windows are too neutral. 2009-10 and 2020 H2-2021 both look more muted than the intended product narrative, so the next config pass should focus on post-shock recovery responsiveness rather than clipping macro inflation back down.

4. **Risk overlay still looks conservative**. GFC stress share is `29%`, 2018 Q4 is `9%`, and April 2025 is `11%`. That may deserve a later pass on risk thresholds or consecutive-day rules, but it is now a secondary question after the baseline fix.


## Decision For This Round

**Do not retune thresholds or layer weights in the same patch.**

The correct move for Q5 is to land the workflow fix, regenerate the calibration artifacts, and treat this as the new baseline. Any weight/threshold pass done before this fix would have been optimizing against the wrong macro scale.

**Recommended Q6 focus**:

* Re-examine `2009-10 Recovery` and `2020 H2-2021 Reopening` under the corrected baseline.
* Test modest market-heavier blends and/or slightly lower axis thresholds, but only against the fixed calibration baseline.
* Separately evaluate whether the independent risk overlay should trigger earlier in GFC / 2018 Q4 without making April 2025 look like a full crisis.


## Validation

* `conda run -n py313 python -m pytest tests/unit/workflows/test_regime_calibration.py -q`
* `conda run -n py313 python -m market_helper.cli.main regime-calibrate`

The generated `regime_v2_calibration_summary.json` now matches the intended macro latent-space behavior and should be the reference point for the next tuning round.
